# 03 — Speed + Memory Benchmarks

Measure tokens/sec and peak memory per parser on both languages. Three repeats, averaged.

In [ ]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import statistics
import pandas as pd

from src.data import load_sentences
from src.parsers import SpacyParser, StanzaParser
from src.perf import measure_throughput, measure_peak_memory

EN_TOKS = [s.tokens for s in load_sentences(Path("../data/en_ewt_test.conllu"))]
RU_TOKS = [s.tokens for s in load_sentences(Path("../data/ru_syntagrus_test.conllu"))]
print(f"EN: {len(EN_TOKS)} sents | RU: {len(RU_TOKS)} sents")

In [ ]:
BATCH_SIZE = 32
REPEATS = 3
WARMUP = 3

def make_batches(tokens, batch_size=BATCH_SIZE):
    return [tokens[i:i+batch_size] for i in range(0, len(tokens), batch_size)]

def bench(parser, tokens):
    batches = make_batches(tokens)
    speeds = []
    for _ in range(REPEATS):
        t = measure_throughput(parser.parse, batches, warmup=WARMUP)
        speeds.append(t.tokens_per_sec)
    mean_tps = statistics.mean(speeds)
    std_tps = statistics.stdev(speeds) if REPEATS > 1 else 0.0
    peak = measure_peak_memory(lambda: parser.parse(tokens[:200]))
    return mean_tps, std_tps, peak

configs = [
    ("en", EN_TOKS, SpacyParser("en_core_web_trf")),
    ("en", EN_TOKS, StanzaParser("en")),
    ("ru", RU_TOKS, SpacyParser("ru_core_news_lg")),
    ("ru", RU_TOKS, StanzaParser("ru")),
]

rows = []
for lang, toks, parser in configs:
    print(f"Benchmarking {lang} {parser.name} ...")
    mean_tps, std_tps, peak = bench(parser, toks)
    rows.append({
        "lang": lang, "parser": parser.name,
        "family": "transition" if "spacy" in parser.name else "graph",
        "tokens_per_sec_mean": round(mean_tps, 1),
        "tokens_per_sec_std": round(std_tps, 1),
        "peak_mb": round(peak / 1e6, 1),
    })
    print(f"  {mean_tps:.0f} tok/s ±{std_tps:.0f}  peak={peak/1e6:.1f} MB")

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("../results/performance.csv", index=False)
print("Saved results/performance.csv")
df

## What we learned
- Expected: spaCy faster (transition-based makes greedy local decisions)
- Expected: Stanza uses more peak memory (biaffine attention scores)
- Ready for error analysis (notebook 04)